# 📱 eidcloud-pwa-db: Quickstart & Integration Notebook

Welcome to the **eidcloud-pwa-db** quickstart notebook. This guide demonstrates how to:
1. Setup PHP runtime environment and inspect the toolchain.
2. Define and validate IndexedDB schemas and compile the client-side JavaScript SDK.
3. Process offline delta mutations and simulate conflict resolution using `SyncGateway`.

In [ ]:
# Cell 1: Environment Inspection & Schema Validation
!php -v
!php bin/eidcloud-pwa-db --help
!php bin/eidcloud-pwa-db validate-schema schema.json

In [ ]:
# Cell 2: Compile the Client-side IndexedDB JavaScript SDK
!php bin/eidcloud-pwa-db compile-sdk --schema=schema.json --out=./dist/
!ls -la ./dist/
!head -n 25 ./dist/eidcloud-pwa-db.js

In [ ]:
# Cell 3: Server-side SyncGateway & Conflict Resolution Simulation
php_script = '''<?php
require_once 'src/Sync/SyncGateway.php';

use EidCloud\\PwaDb\\Sync\\SyncGateway;

$gateway = new SyncGateway(['tables' => ['customers' => ['primaryKey' => 'id']]]);

$payload = [
    'clientId' => 'pwa-client-007',
    'timestamp' => time(),
    'transactions' => [
        [
            'id' => 'tx-colab-1',
            'table' => 'customers',
            'action' => 'insert',
            'recordId' => 'cust-1',
            'data' => ['name' => 'Sara Connor', 'status' => 'active'],
            'timestamp' => time()
        ]
    ]
];

$response = $gateway->processBatch($payload);
echo json_encode($response, JSON_PRETTY_PRINT | JSON_UNESCAPED_SLASHES) . PHP_EOL;
'''
with open('simulate_sync.php', 'w') as f:
    f.write(php_script)

!php simulate_sync.php
!rm simulate_sync.php